# Add a Simple Personalized Recommendation System based on Genres that a User Likes

In [1]:
from pathlib import Path
import sys

current_path = Path.cwd().resolve()

for path in [current_path, *current_path.parents]:
    if (path / "src" / "data.py").exists():
        project_root = path
        break
else:
    raise FileNotFoundError("Could not find project root containing src/data.py")

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd

from src.data import load_movies
from src.recommenders import recommend_popular_movies

train = pd.read_csv(project_root / "data/processed/train_ratings.csv")
test = pd.read_csv(project_root / "data/processed/test_ratings.csv")
movies = load_movies()


In [2]:
from src.recommenders import build_user_genre_profile

build_user_genre_profile(train, movies, user_id=1).head(10)

,genre,positive_count
0,Adventure,71
1,Action,68
2,Comedy,58
3,Drama,46
4,Children,37
5,Fantasy,36
6,Crime,35
7,Thriller,34
8,Animation,27
9,Sci-Fi,24


# Personalized Recommendation Based on Genre Evaluation (Naive, not hybrid)
* Compare how recommendation based on which genre each user likes to the baseline recommendation system based on popularity. 


In [3]:
from src.recommenders import recommend_genre_personalized_movies

recommend_genre_personalized_movies(train, movies, user_id=1, k=10)

,movieId,title,genres,genre_score,positive_ratings
0,81132,Rubber (2010),Action|Adventure|Comedy|Crime|Drama|Film-Noir|...,325,1
1,117646,Dragonheart 2: A New Beginning (2000),Action|Adventure|Comedy|Drama|Fantasy|Thriller,313,1
2,108932,The Lego Movie (2014),Action|Adventure|Animation|Children|Comedy|Fan...,297,14
3,26340,"Twelve Tasks of Asterix, The (Les douze travau...",Action|Adventure|Animation|Children|Comedy|Fan...,297,2
4,51939,TMNT (Teenage Mutant Ninja Turtles) (2007),Action|Adventure|Animation|Children|Comedy|Fan...,297,1
5,71999,Aelita: The Queen of Mars (Aelita) (1924),Action|Adventure|Drama|Fantasy|Romance|Sci-Fi|...,295,1
6,546,Super Mario Bros. (1993),Action|Adventure|Children|Comedy|Fantasy|Sci-Fi,294,0
7,4956,"Stunt Man, The (1980)",Action|Adventure|Comedy|Drama|Romance|Thriller,293,1
8,52462,Aqua Teen Hunger Force Colon Movie Film for Th...,Action|Adventure|Animation|Comedy|Fantasy|Myst...,292,0
9,164226,Maximum Ride (2016),Action|Adventure|Comedy|Fantasy|Sci-Fi|Thriller,291,1


In [4]:
recommend_popular_movies(train, movies, user_id=1, k=10)



,movieId,title,genres,positive_ratings
0,318,"Shawshank Redemption, The (1994)",Crime|Drama,240
1,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,205
2,527,Schindler's List (1993),Drama|War,146
3,858,"Godfather, The (1972)",Crime|Drama,144
4,4993,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,138
5,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi,134
6,7153,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy,133
7,5952,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy,121
8,47,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,118
9,150,Apollo 13 (1995),Adventure|Drama|IMAX,117


The recommendations now differ significantly from the popularity based recommendations. 

# Precision@K, Recall@K, NDCG@K

In [5]:
from src.recommenders import recommend_genre_personalized_movies
from src.metrics import (
    precision_at_k,
    recall_at_k,
    ndcg_at_k,
    rated_precision_at_k,
)

In [6]:
k = 10
genre_scores = []

test_users = test["userId"].unique()

for user_id in test_users:
    recs = recommend_genre_personalized_movies(
        train_ratings=train,
        movies=movies,
        user_id=user_id,
        k=k,
    )

    recommended_items = recs["movieId"].tolist()

    user_test = test[test["userId"] == user_id]
    rated_items = set(user_test["movieId"])
    relevant_items = set(user_test[user_test["is_positive"]]["movieId"])

    precision = precision_at_k(
        recommended_items=recommended_items,
        relevant_items=relevant_items,
        k=k,
    )

    recall = recall_at_k(
        recommended_items=recommended_items,
        relevant_items=relevant_items,
        k=k,
    )

    ndcg = ndcg_at_k(
        recommended_items=recommended_items,
        relevant_items=relevant_items,
        k=k,
    )

    rated_precision = rated_precision_at_k(
        recommended_items=recommended_items,
        positive_items=relevant_items,
        rated_items=rated_items,
        k=k,
    )

    genre_scores.append(
        {
            "userId": user_id,
            "precision_at_10": precision,
            "recall_at_10": recall,
            "ndcg_at_10": ndcg,
            "rated_precision_at_10": rated_precision,
            "n_future_likes": len(relevant_items),
        }
    )

genre_results = pd.DataFrame(genre_scores)

genre_results.describe()

,userId,precision_at_10,recall_at_10,ndcg_at_10,rated_precision_at_10,n_future_likes
count,610.000000,610.000000,610.000000,610.000000,46.000000,610.000000
mean,305.500000,0.004262,0.005524,0.005436,0.554348,15.134426
std,176.236111,0.020217,0.038025,0.031440,0.496972,22.249480
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,153.250000,0.000000,0.000000,0.000000,0.000000,4.000000
50%,305.500000,0.000000,0.000000,0.000000,1.000000,7.000000
75%,457.750000,0.000000,0.000000,0.000000,1.000000,17.000000
max,610.000000,0.100000,0.500000,0.390380,1.000000,241.000000


The results are much lower than the popular baseline. It appears as if we can't rely entirely on genre but need some popularity as well. Therefore, a hybrid score might work better. Also, the current scores for each genre may not be the most efficient. The current setup tends to overreward multi genre movies. This approach also might be too genre specific, that is, it might give a user recommendations only based on their favorite genre. A better approach might be to make recommendations proportionally to the distribution of genres that a user rates positively. 

It appears as if a stronger mix of components is needed. This hybrid system might consist of:
* Genre match score
* Popularity score
* Average rating score (better idea if popular movies are actually liked or just popular)
* Recency score
* Semantic score (based on descriptions of these movies, can we discover hidden themes that a user might like)
* Recommendation diversity (not just purely recommending one genre but a mix of genres if possible)

Right now, the focus is going to be implementing the genre match score and the popularity score mix, and then expanding later. 


# Hybrid Genre and Popularity Recommendation

In [7]:
from src.recommenders import recommend_hybrid_genre_popularity_movies

recommend_hybrid_genre_popularity_movies(train, movies, user_id=1, k=10)

,movieId,title,genres,genre_match_score,popularity_score,hybrid_score,positive_ratings
0,2918,Ferris Bueller's Day Off (1986),Comedy,1.0,0.737138,0.894855,56.0
1,2791,Airplane! (1980),Comedy,1.0,0.709565,0.883826,48.0
2,1288,This Is Spinal Tap (1984),Comedy,1.0,0.689942,0.875977,43.0
3,344,Ace Ventura: Pet Detective (1994),Comedy,1.0,0.681460,0.872584,41.0
4,141,"Birdcage, The (1996)",Comedy,1.0,0.663213,0.865285,37.0
5,3948,Meet the Parents (2000),Comedy,1.0,0.658350,0.863340,36.0
6,1394,Raising Arizona (1987),Comedy,1.0,0.648219,0.859287,34.0
7,3421,Animal House (1978),Comedy,1.0,0.648219,0.859287,34.0
8,104,Happy Gilmore (1996),Comedy,1.0,0.642934,0.857173,33.0
9,2599,Election (1999),Comedy,1.0,0.631880,0.852752,31.0


These recommendations look more reasonable now. Maybe the recommendation mix might need some work in future versions but it's fine for now. Let's see how this version compares to the baseline. 

In [8]:
k = 10
hybrid_scores = []

test_users = test["userId"].unique()

for user_id in test_users:
    recs = recommend_hybrid_genre_popularity_movies(
        train_ratings=train,
        movies=movies,
        user_id=user_id,
        k=k,
    )

    recommended_items = recs["movieId"].tolist()

    user_test = test[test["userId"] == user_id]
    rated_items = set(user_test["movieId"])
    relevant_items = set(user_test[user_test["is_positive"]]["movieId"])

    precision = precision_at_k(
        recommended_items=recommended_items,
        relevant_items=relevant_items,
        k=k,
    )

    recall = recall_at_k(
        recommended_items=recommended_items,
        relevant_items=relevant_items,
        k=k,
    )

    ndcg = ndcg_at_k(
        recommended_items=recommended_items,
        relevant_items=relevant_items,
        k=k,
    )

    rated_precision = rated_precision_at_k(
        recommended_items=recommended_items,
        positive_items=relevant_items,
        rated_items=rated_items,
        k=k,
    )

    hybrid_scores.append(
        {
            "userId": user_id,
            "precision_at_10": precision,
            "recall_at_10": recall,
            "ndcg_at_10": ndcg,
            "rated_precision_at_10": rated_precision,
            "n_future_likes": len(relevant_items),
        }
    )

hybrid_results = pd.DataFrame(hybrid_scores)

hybrid_results.describe()


,userId,precision_at_10,recall_at_10,ndcg_at_10,rated_precision_at_10,n_future_likes
count,610.000000,610.000000,610.000000,610.000000,153.000000,610.000000
mean,305.500000,0.023115,0.017132,0.030171,0.655447,15.134426
std,176.236111,0.057651,0.057885,0.080072,0.448763,22.249480
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,153.250000,0.000000,0.000000,0.000000,0.000000,4.000000
50%,305.500000,0.000000,0.000000,0.000000,1.000000,7.000000
75%,457.750000,0.000000,0.000000,0.000000,1.000000,17.000000
max,610.000000,0.400000,0.750000,0.831872,1.000000,241.000000


In [9]:
def evaluate_recommender(recommender_function, model_name: str) -> dict:
    scores = []

    for user_id in test["userId"].unique():
        recs = recommender_function(
            train_ratings=train,
            movies=movies,
            user_id=user_id,
            k=10,
        )

        recommended_items = recs["movieId"].tolist()

        user_test = test[test["userId"] == user_id]
        rated_items = set(user_test["movieId"])
        positive_items = set(user_test[user_test["is_positive"]]["movieId"])

        scores.append(
            {
                "precision_at_10": precision_at_k(
                    recommended_items=recommended_items,
                    relevant_items=positive_items,
                    k=10,
                ),
                "recall_at_10": recall_at_k(
                    recommended_items=recommended_items,
                    relevant_items=positive_items,
                    k=10,
                ),
                "ndcg_at_10": ndcg_at_k(
                    recommended_items=recommended_items,
                    relevant_items=positive_items,
                    k=10,
                ),
                "rated_precision_at_10": rated_precision_at_k(
                    recommended_items=recommended_items,
                    positive_items=positive_items,
                    rated_items=rated_items,
                    k=10,
                ),
            }
        )

    results = pd.DataFrame(scores)

    return {
        "model": model_name,
        "precision_at_10": results["precision_at_10"].mean(),
        "recall_at_10": results["recall_at_10"].mean(),
        "ndcg_at_10": results["ndcg_at_10"].mean(),
        "rated_precision_at_10": results["rated_precision_at_10"].mean(),
        "rated_precision_users": results["rated_precision_at_10"].notna().sum(),
    }

comparison = pd.DataFrame(
    [
        evaluate_recommender(recommend_popular_movies, "popularity"),
        evaluate_recommender(recommend_genre_personalized_movies, "naive_genre"),
        evaluate_recommender(
            recommend_hybrid_genre_popularity_movies,
            "hybrid_genre_popularity",
        ),
    ]
)

comparison

,model,precision_at_10,recall_at_10,ndcg_at_10,rated_precision_at_10,rated_precision_users
0,popularity,0.057869,0.049672,0.075906,0.771876,221
1,naive_genre,0.004262,0.005524,0.005436,0.554348,46
2,hybrid_genre_popularity,0.023115,0.017132,0.030171,0.655447,153


# Genre-Boosted Popularity Recommendation

This model keeps popularity as the main ranking signal, then applies a smaller boost for movies whose genres match the user's positive rating history. This is a more conservative personalization strategy than the earlier hybrid model.

In [10]:
from src.recommenders import recommend_genre_boosted_popular_movies

recommend_genre_boosted_popular_movies(train, movies, user_id=1, k=10)

,movieId,title,genres,popularity_score,genre_match_score,boosted_score,positive_ratings
0,318,"Shawshank Redemption, The (1994)",Crime|Drama,1.000000,0.550626,0.865188,240.0
1,7153,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy,0.892985,0.685093,0.830617,133.0
2,4993,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,0.899664,0.622720,0.816581,138.0
3,2918,Ferris Bueller's Day Off (1986),Comedy,0.737138,1.000000,0.815996,56.0
4,1270,Back to the Future (1985),Adventure|Comedy|Sci-Fi,0.851960,0.713746,0.810496,106.0
5,858,"Godfather, The (1972)",Crime|Drama,0.907369,0.550626,0.800346,144.0
6,5952,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy,0.875879,0.622720,0.799932,121.0
7,2791,Airplane! (1980),Comedy,0.709565,1.000000,0.796696,48.0
8,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi,0.894340,0.563018,0.794944,134.0
9,6539,Pirates of the Caribbean: The Curse of the Bla...,Action|Adventure|Comedy|Fantasy,0.801205,0.770154,0.791890,80.0


# Final Model Comparison with Rated-Only Precision

This comparison includes the standard offline ranking metrics and a complementary rated-only precision metric. Rated-only precision evaluates only recommendations that appear in the user's later ratings, so missing recommendation feedback is treated as unknown rather than automatically wrong.

In [11]:
from src.metrics import (
    precision_at_k,
    recall_at_k,
    ndcg_at_k,
    rated_precision_at_k,
)
from src.recommenders import (
    recommend_popular_movies,
    recommend_genre_personalized_movies,
    recommend_hybrid_genre_popularity_movies,
    recommend_genre_boosted_popular_movies,
)

def evaluate_recommender(recommender_function, model_name: str, k: int = 10) -> dict:
    user_scores = []

    for user_id in test["userId"].unique():
        recs = recommender_function(
            train_ratings=train,
            movies=movies,
            user_id=user_id,
            k=k,
        )

        recommended_items = recs["movieId"].tolist()

        user_test = test[test["userId"] == user_id]
        rated_items = set(user_test["movieId"])
        positive_items = set(user_test[user_test["is_positive"]]["movieId"])

        user_scores.append(
            {
                "precision_at_10": precision_at_k(
                    recommended_items=recommended_items,
                    relevant_items=positive_items,
                    k=k,
                ),
                "recall_at_10": recall_at_k(
                    recommended_items=recommended_items,
                    relevant_items=positive_items,
                    k=k,
                ),
                "ndcg_at_10": ndcg_at_k(
                    recommended_items=recommended_items,
                    relevant_items=positive_items,
                    k=k,
                ),
                "rated_precision_at_10": rated_precision_at_k(
                    recommended_items=recommended_items,
                    positive_items=positive_items,
                    rated_items=rated_items,
                    k=k,
                ),
            }
        )

    results = pd.DataFrame(user_scores)

    return {
        "model": model_name,
        "precision_at_10": results["precision_at_10"].mean(),
        "recall_at_10": results["recall_at_10"].mean(),
        "ndcg_at_10": results["ndcg_at_10"].mean(),
        "rated_precision_at_10": results["rated_precision_at_10"].mean(),
        "users_with_rated_recommendations": results["rated_precision_at_10"].notna().sum(),
    }

model_comparison = pd.DataFrame(
    [
        evaluate_recommender(recommend_popular_movies, "popularity"),
        evaluate_recommender(recommend_genre_personalized_movies, "naive_genre"),
        evaluate_recommender(
            recommend_hybrid_genre_popularity_movies,
            "hybrid_genre_popularity",
        ),
        evaluate_recommender(
            recommend_genre_boosted_popular_movies,
            "genre_boosted_popularity",
        ),
    ]
)

model_comparison


,model,precision_at_10,recall_at_10,ndcg_at_10,rated_precision_at_10,users_with_rated_recommendations
0,popularity,0.057869,0.049672,0.075906,0.771876,221
1,naive_genre,0.004262,0.005524,0.005436,0.554348,46
2,hybrid_genre_popularity,0.023115,0.017132,0.030171,0.655447,153
3,genre_boosted_popularity,0.049508,0.043402,0.065751,0.757164,229


It appears as if the baseline popularity model is still performing the best. It might be time to make some stronger changes to our recommendation system. It might work best to prioritize popularity and then tailor and boost movies to a user' taste. 

# Product Quality Diagnostics

The offline ranking metrics mostly reward matching future observed ratings. To understand product tradeoffs, we also compare catalog coverage, recommendation popularity, and genre diversity. These diagnostics help answer whether personalization offers more discovery or variety even when popularity performs best on relevance metrics.

In [12]:
def collect_recommendations(recommender_function, model_name: str, k: int = 10) -> pd.DataFrame:
    recommendation_rows = []

    positive_counts = (
        train[train["is_positive"]]
        .groupby("movieId")
        .size()
        .reset_index(name="global_positive_ratings")
    )

    for user_id in test["userId"].unique():
        recs = recommender_function(
            train_ratings=train,
            movies=movies,
            user_id=user_id,
            k=k,
        ).copy()

        recs["userId"] = user_id
        recs["rank"] = range(1, len(recs) + 1)
        recs["model"] = model_name
        recommendation_rows.append(recs)

    recommendations = pd.concat(recommendation_rows, ignore_index=True)

    recommendations = recommendations.merge(
        positive_counts,
        on="movieId",
        how="left",
    )

    recommendations["global_positive_ratings"] = (
        recommendations["global_positive_ratings"].fillna(0)
    )

    return recommendations

popularity_recs = collect_recommendations(
    recommend_popular_movies,
    "popularity",
    k=10,
)

boosted_recs = collect_recommendations(
    recommend_genre_boosted_popular_movies,
    "genre_boosted_popularity",
    k=10,
)

all_recs = pd.concat([popularity_recs, boosted_recs], ignore_index=True)

all_recs.head()


,movieId,title,genres,positive_ratings,userId,rank,model,global_positive_ratings,popularity_score,genre_match_score,boosted_score
0,318,"Shawshank Redemption, The (1994)",Crime|Drama,240.0,1,1,popularity,240,NaN,NaN,NaN
1,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,205.0,1,2,popularity,205,NaN,NaN,NaN
2,527,Schindler's List (1993),Drama|War,146.0,1,3,popularity,146,NaN,NaN,NaN
3,858,"Godfather, The (1972)",Crime|Drama,144.0,1,4,popularity,144,NaN,NaN,NaN
4,4993,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,138.0,1,5,popularity,138,NaN,NaN,NaN


In [13]:
def count_unique_genres(genres: pd.Series) -> int:
    unique_genres = set()

    for genre_string in genres.dropna():
        for genre in genre_string.split("|"):
            unique_genres.add(genre)

    return len(unique_genres)

def summarize_product_diagnostics(recommendations: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for model_name, model_recs in recommendations.groupby("model"):
        unique_movies = model_recs["movieId"].nunique()
        catalog_coverage = unique_movies / movies["movieId"].nunique()

        genres_per_user = (
            model_recs.groupby("userId")["genres"]
            .apply(count_unique_genres)
        )

        rows.append(
            {
                "model": model_name,
                "unique_recommended_movies": unique_movies,
                "catalog_coverage": catalog_coverage,
                "avg_global_positive_ratings": model_recs["global_positive_ratings"].mean(),
                "median_global_positive_ratings": model_recs["global_positive_ratings"].median(),
                "avg_unique_genres_per_user": genres_per_user.mean(),
            }
        )

    return pd.DataFrame(rows)

product_diagnostics = summarize_product_diagnostics(all_recs)

product_diagnostics


,model,unique_recommended_movies,catalog_coverage,avg_global_positive_ratings,median_global_positive_ratings,avg_unique_genres_per_user
0,genre_boosted_popularity,137,0.014063,114.829836,100.0,6.701639
1,popularity,91,0.009341,165.132131,155.0,10.813115


Catalog coverage is slight better the genre boosted popularity model. Unique genres per user is slightly less though. This may not necessarily be a bad thing though, since some users are more fixated on a certain couple of genres than a wide variety of genres. 

The next step will be implementing semantic metadeta into our recommendation system.

# Content Based Recommendations

In [14]:
from src.content import build_movie_content_features
from src.recommenders import (
    build_content_tfidf_matrix,
    recommend_content_based_movies_from_vectors,
    recommend_true_hybrid_movies_from_vectors,
)


In [15]:
tmdb_metadata = pd.read_csv("../data/processed/tmdb_movie_metadata.csv")

movie_content = build_movie_content_features(
    movies=movies,
    tmdb_metadata=tmdb_metadata,
)

_, content_vectors = build_content_tfidf_matrix(movie_content)

movie_content.head()


,movieId,title,genres,tmdb_title,overview,tagline,tmdb_genres,content_text
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story,"Led by Woody, Andy's toys live happily in his ...",The adventure takes off when toys come to life!,Family|Comedy|Animation|Adventure,Toy Story (1995) Adventure Animation Children ...
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji,When siblings Judy and Peter discover an encha...,It's a jungle in here.,Adventure|Fantasy|Family,Jumanji (1995) Adventure Children Fantasy Adve...
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men,A family wedding reignites the ancient feud be...,Still Yelling. Still Fighting. Still Ready for...,Romance|Comedy,Grumpier Old Men (1995) Comedy Romance Romance...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",Friends are the people who let you be yourself...,Comedy|Drama|Romance,Waiting to Exhale (1995) Comedy Drama Romance ...
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II,Just when George Banks has recovered from his ...,Just when his world is back to normal... he's ...,Comedy|Family,Father of the Bride Part II (1995) Comedy Come...


In [16]:
content_recs = recommend_content_based_movies_from_vectors(
    train_ratings=train,
    movie_content=movie_content,
    content_vectors=content_vectors,
    user_id=1,
    k=10,
)

content_recs


,movieId,title,genres,content_similarity
0,27036,Merlin (1998),Action|Adventure|Drama|Fantasy|Romance,0.446234
1,5069,Escaflowne: The Movie (Escaflowne) (2000),Action|Adventure|Animation|Drama|Fantasy,0.417387
2,26510,"Ewok Adventure, The (a.k.a. Caravan of Courage...",Adventure|Children|Fantasy|Sci-Fi,0.402125
3,148775,Wizards of Waverly Place: The Movie (2009),Adventure|Children|Comedy|Drama|Fantasy|Sci-Fi,0.375481
4,631,All Dogs Go to Heaven 2 (1996),Adventure|Animation|Children|Fantasy|Musical|R...,0.362211
5,3054,Pokémon: The First Movie (1998),Adventure|Animation|Children|Fantasy|Sci-Fi,0.359693
6,139130,Afro Samurai (2007),Action|Adventure|Animation|Drama|Fantasy,0.344153
7,55995,Beowulf (2007),Action|Adventure|Animation|Fantasy|IMAX,0.343557
8,172547,Despicable Me 3 (2017),Adventure|Animation|Children|Comedy,0.339876
9,610,Heavy Metal (1981),Action|Adventure|Animation|Horror|Sci-Fi,0.336073


In [17]:
def recommend_content_based_wrapper(
    train_ratings: pd.DataFrame,
    movies: pd.DataFrame,
    user_id: int,
    k: int = 10,
) -> pd.DataFrame:
    return recommend_content_based_movies_from_vectors(
        train_ratings=train_ratings,
        movie_content=movie_content,
        content_vectors=content_vectors,
        user_id=user_id,
        k=k,
    )


def recommend_true_hybrid_wrapper(
    train_ratings: pd.DataFrame,
    movies: pd.DataFrame,
    user_id: int,
    k: int = 10,
) -> pd.DataFrame:
    return recommend_true_hybrid_movies_from_vectors(
        train_ratings=train_ratings,
        movie_content=movie_content,
        content_vectors=content_vectors,
        user_id=user_id,
        k=k,
    )

new_model_evaluations = pd.DataFrame(
    [
        evaluate_recommender(
            recommend_content_based_wrapper,
            "content_based",
        ),
        evaluate_recommender(
            recommend_true_hybrid_wrapper,
            "true_hybrid",
        ),
    ]
)

model_comparison_with_content = pd.concat(
    [model_comparison, new_model_evaluations],
    ignore_index=True,
)

model_comparison_with_content


,model,precision_at_10,recall_at_10,ndcg_at_10,rated_precision_at_10,users_with_rated_recommendations
0,popularity,0.057869,0.049672,0.075906,0.771876,221
1,naive_genre,0.004262,0.005524,0.005436,0.554348,46
2,hybrid_genre_popularity,0.023115,0.017132,0.030171,0.655447,153
3,genre_boosted_popularity,0.049508,0.043402,0.065751,0.757164,229
4,content_based,0.006066,0.007579,0.008429,0.630719,51
5,true_hybrid,0.049508,0.043598,0.061609,0.774364,236


* Now the true hybrid is beating the popular baseline on our main metric, rated_precious_at_10 and is comparable on precision@10, recall@10, ndcg@10, while also reaching more users with rated recommendations. 

### Catalog and Product Diagnostics

These metrics show whether each recommender is broadening the catalog or mostly recommending the same popular movies to everyone.


In [18]:
def collect_model_recommendations(
    recommender_function,
    model_name: str,
    k: int = 10,
) -> pd.DataFrame:
    recommendation_rows = []

    positive_counts = (
        train[train["is_positive"]]
        .groupby("movieId")
        .size()
        .reset_index(name="global_positive_ratings")
    )

    for user_id in test["userId"].unique():
        recs = recommender_function(
            train_ratings=train,
            movies=movies,
            user_id=user_id,
            k=k,
        ).copy()

        if recs.empty:
            continue

        recs["userId"] = user_id
        recs["rank"] = range(1, len(recs) + 1)
        recs["model"] = model_name
        recommendation_rows.append(recs)

    recommendations = pd.concat(recommendation_rows, ignore_index=True)

    recommendations = recommendations.merge(
        positive_counts,
        on="movieId",
        how="left",
    )

    recommendations["global_positive_ratings"] = (
        recommendations["global_positive_ratings"].fillna(0)
    )

    return recommendations


def count_unique_genres(genres: pd.Series) -> int:
    unique_genres = set()

    for genre_string in genres.dropna():
        for genre in genre_string.split("|"):
            unique_genres.add(genre)

    return len(unique_genres)


def summarize_catalog_diagnostics(recommendations: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for model_name, model_recs in recommendations.groupby("model"):
        unique_movies = model_recs["movieId"].nunique()
        catalog_coverage = unique_movies / movies["movieId"].nunique()

        genres_per_user = (
            model_recs.groupby("userId")["genres"]
            .apply(count_unique_genres)
        )

        rows.append(
            {
                "model": model_name,
                "unique_recommended_movies": unique_movies,
                "catalog_coverage": catalog_coverage,
                "avg_global_positive_ratings": model_recs["global_positive_ratings"].mean(),
                "median_global_positive_ratings": model_recs["global_positive_ratings"].median(),
                "avg_unique_genres_per_user": genres_per_user.mean(),
            }
        )

    return pd.DataFrame(rows).sort_values(
        "catalog_coverage",
        ascending=False,
    )


catalog_recommendations = pd.concat(
    [
        collect_model_recommendations(
            recommend_popular_movies,
            "popularity",
            k=10,
        ),
        collect_model_recommendations(
            recommend_genre_boosted_popular_movies,
            "genre_boosted_popularity",
            k=10,
        ),
        collect_model_recommendations(
            recommend_content_based_wrapper,
            "content_based",
            k=10,
        ),
        collect_model_recommendations(
            recommend_true_hybrid_wrapper,
            "true_hybrid",
            k=10,
        ),
    ],
    ignore_index=True,
)

catalog_diagnostics = summarize_catalog_diagnostics(catalog_recommendations)

catalog_diagnostics


,model,unique_recommended_movies,catalog_coverage,avg_global_positive_ratings,median_global_positive_ratings,avg_unique_genres_per_user
0,content_based,899,0.092281,3.902142,1.0,10.364086
3,true_hybrid,424,0.043523,112.212459,97.0,9.872131
1,genre_boosted_popularity,137,0.014063,114.829836,100.0,6.701639
2,popularity,91,0.009341,165.132131,155.0,10.813115


The true hybrid significantly increases catalog coverage from the popularity baseline. In fact the true hybrid covers 4.66 times more of the catalog, which from a product sense is significantly better. 

### Hybrid Weight Sensitivity Analysis

The true hybrid recommender has three tunable signals: popularity, content similarity, and genre preference. This section tests several weight combinations to see how the product tradeoff changes.


In [19]:
hybrid_weight_grid = [
    {"popularity_weight": 0.80, "content_weight": 0.10, "genre_weight": 0.10},
    {"popularity_weight": 0.70, "content_weight": 0.20, "genre_weight": 0.10},
    {"popularity_weight": 0.60, "content_weight": 0.30, "genre_weight": 0.10},
    {"popularity_weight": 0.55, "content_weight": 0.30, "genre_weight": 0.15},
    {"popularity_weight": 0.50, "content_weight": 0.40, "genre_weight": 0.10},
    {"popularity_weight": 0.50, "content_weight": 0.30, "genre_weight": 0.20},
    {"popularity_weight": 0.40, "content_weight": 0.50, "genre_weight": 0.10},
]


def evaluate_hybrid_weight_setting(weights: dict, k: int = 10) -> dict:
    user_scores = []
    recommendation_rows = []

    positive_counts = (
        train[train["is_positive"]]
        .groupby("movieId")
        .size()
        .reset_index(name="global_positive_ratings")
    )

    for user_id in test["userId"].unique():
        recs = recommend_true_hybrid_movies_from_vectors(
            train_ratings=train,
            movie_content=movie_content,
            content_vectors=content_vectors,
            user_id=user_id,
            k=k,
            popularity_weight=weights["popularity_weight"],
            content_weight=weights["content_weight"],
            genre_weight=weights["genre_weight"],
        ).copy()

        recommended_items = recs["movieId"].tolist()

        user_test = test[test["userId"] == user_id]
        rated_items = set(user_test["movieId"])
        positive_items = set(user_test[user_test["is_positive"]]["movieId"])

        user_scores.append(
            {
                "precision_at_10": precision_at_k(
                    recommended_items=recommended_items,
                    relevant_items=positive_items,
                    k=k,
                ),
                "recall_at_10": recall_at_k(
                    recommended_items=recommended_items,
                    relevant_items=positive_items,
                    k=k,
                ),
                "ndcg_at_10": ndcg_at_k(
                    recommended_items=recommended_items,
                    relevant_items=positive_items,
                    k=k,
                ),
                "rated_precision_at_10": rated_precision_at_k(
                    recommended_items=recommended_items,
                    positive_items=positive_items,
                    rated_items=rated_items,
                    k=k,
                ),
            }
        )

        recs["userId"] = user_id
        recs["rank"] = range(1, len(recs) + 1)
        recommendation_rows.append(recs)

    score_results = pd.DataFrame(user_scores)

    recommendations = pd.concat(recommendation_rows, ignore_index=True)
    recommendations = recommendations.merge(
        positive_counts,
        on="movieId",
        how="left",
    )
    recommendations["global_positive_ratings"] = (
        recommendations["global_positive_ratings"].fillna(0)
    )

    return {
        "weights": (
            f"pop={weights['popularity_weight']:.2f}, "
            f"content={weights['content_weight']:.2f}, "
            f"genre={weights['genre_weight']:.2f}"
        ),
        "precision_at_10": score_results["precision_at_10"].mean(),
        "recall_at_10": score_results["recall_at_10"].mean(),
        "ndcg_at_10": score_results["ndcg_at_10"].mean(),
        "rated_precision_at_10": score_results["rated_precision_at_10"].mean(),
        "users_with_rated_recommendations": score_results["rated_precision_at_10"].notna().sum(),
        "unique_recommended_movies": recommendations["movieId"].nunique(),
        "catalog_coverage": recommendations["movieId"].nunique() / movies["movieId"].nunique(),
        "avg_global_positive_ratings": recommendations["global_positive_ratings"].mean(),
    }


hybrid_weight_results = pd.DataFrame(
    [
        evaluate_hybrid_weight_setting(weights)
        for weights in hybrid_weight_grid
    ]
)

hybrid_weight_results.sort_values(
    ["ndcg_at_10", "catalog_coverage"],
    ascending=[False, False],
)


,weights,precision_at_10,recall_at_10,ndcg_at_10,rated_precision_at_10,users_with_rated_recommendations,unique_recommended_movies,catalog_coverage,avg_global_positive_ratings
0,"pop=0.80, content=0.10, genre=0.10",0.060164,0.053978,0.078204,0.775188,240,99,0.010162,160.279344
1,"pop=0.70, content=0.20, genre=0.10",0.057869,0.051663,0.072307,0.775166,251,175,0.017963,149.952131
2,"pop=0.60, content=0.30, genre=0.10",0.051475,0.046471,0.063332,0.770055,241,351,0.036030,126.086230
3,"pop=0.55, content=0.30, genre=0.15",0.049508,0.043598,0.061609,0.774364,236,424,0.043523,112.212459
5,"pop=0.50, content=0.30, genre=0.20",0.044590,0.039485,0.056770,0.780694,221,476,0.048861,95.401475
4,"pop=0.50, content=0.40, genre=0.10",0.041475,0.039102,0.050858,0.754955,222,657,0.067440,93.952131
6,"pop=0.40, content=0.50, genre=0.10",0.032459,0.031993,0.040186,0.705471,198,825,0.084685,67.917049


It appears as if we chose a fairly good tradeoff on the first choice. Yes, increasing the weights to pop=0.80, content=0.10, and genre=0.10 increases the accuracy on some of our recommendation metrics but it decreases the catalog coverage by nearly 4 times. In this scenario, the pop=0.50, content=0.30, genre=0.20 might be the best tradeoff with the highest rated_precision_at_10 and still a high catalog coverage. 